<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 06. R-GCN: 관계 종류별 메시지

## Modeling Relational Data with Graph Convolutional Networks

- **원 논문:** [Modeling Relational Data with Graph Convolutional Networks](https://arxiv.org/abs/1703.06103)
- **저자 / 발표:** Michael Schlichtkrull, Thomas N. Kipf, Peter Bloem, Rianne van den Berg, Ivan Titov, Max Welling · ESWC (2018)
- **난이도 / 예상 시간:** 중급 · 약 60분
- **선수 지식:** GCN, knowledge graph, relation types
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 관계별 weight와 degree normalization을 갖는 R-GCN propagation을 구현합니다.

**축소하거나 생략한 부분:** AIFB/MUTAG/BGS/AM와 FB15k 대신 12개 entity·3 relation의 로컬 지식그래프를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (2) | relation-specific neighborhood sum | 관계별 weight·shape |
| Eq. (2)의 c_i,r | 관계별 degree normalization | 행합/isolated 처리 |
| Eq. (3) | basis decomposition | parameter reconstruction |
| Figure 2 | edge type 변화의 메시지 영향 | embedding displacement |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$h_i^{l+1}=\sigma\!\left(\sum_{r\in\mathcal R}\sum_{j\in\mathcal N_i^r}\frac{1}{c_{i
,r}}W_r^l h_j^l+W_0^lh_i^l\right)$$

- **기호 정의:** r은 관계, N_i^r은 관계별 이웃, c는 정규화 상수, W_r은 관계별 weight입니다.
- **수식의 역할:** 관계 종류마다 다른 메시지 변환을 적용해 다중 관계 그래프를 표현합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §2, Eq. (2)을 구조화합니다.
- **입출력 shape:** 관계 인접행렬 [R, N, N], 특징 [N, F] → 노드 표현 [N, H]
- **평가:** 관계별 edge reconstruction BCE와 관계 변경에 따른 표현 변화
- **원문 대비 한계:** AIFB/MUTAG/BGS/AM와 FB15k 대신 12개 entity·3 relation의 로컬 지식그래프를 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §2, Eq. (2) / Eq. (2)의 c_i,r
- **이 셀의 책임:** relation-specific neighborhood sum / 관계별 degree normalization
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** 관계별 weight·shape / 행합/isolated 처리. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
relation_adjacency = torch.tensor(raw["relation_adjacency"], dtype=torch.float32)
relation_features = torch.tensor(raw["relation_features"], dtype=torch.float32)
assert relation_adjacency.ndim == 3 and relation_adjacency.shape[
    1
] == relation_adjacency.shape[2] == len(relation_features)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** Eq. (2)의 c_i,r
- **이 셀의 책임:** 관계별 degree normalization
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 행합/isolated 처리. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def relation_normalize(rel_adj: Tensor) -> Tensor:
    degree = rel_adj.sum(2, keepdim=True).clamp_min(1)
    return rel_adj / degree


rel_norm = relation_normalize(relation_adjacency)
row_sums = rel_norm.sum(2)
assert ((row_sums == 0) | (torch.isclose(row_sums, torch.ones_like(row_sums)))).all()

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** Eq. (3)
- **이 셀의 책임:** basis decomposition
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** parameter reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def basis_weights(bases: Tensor, coefficients: Tensor) -> Tensor:
    return torch.einsum("rb,bio->rio", coefficients, bases)


bases = torch.randn(2, relation_features.shape[1], 8) * 0.2
coeff = torch.randn(len(rel_norm), 2)
rel_weights = basis_weights(bases, coeff)
assert rel_weights.shape == (len(rel_norm), relation_features.shape[1], 8)

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2)
- **이 셀의 책임:** relation-specific neighborhood sum
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 관계별 weight·shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def rgcn_layer(x: Tensor, rel_a: Tensor, rel_w: Tensor, self_w: Tensor) -> Tensor:
    messages = torch.stack([rel_a[r] @ x @ rel_w[r] for r in range(len(rel_a))]).sum(0)
    return F.relu(messages + x @ self_w)


self_w = torch.randn(relation_features.shape[1], 8) * 0.2
h = rgcn_layer(relation_features, rel_norm, rel_weights, self_w)
assert h.shape == (len(relation_features), 8)

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** Figure 2
- **이 셀의 책임:** edge type 변화의 메시지 영향
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** embedding displacement. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
changed = relation_adjacency.clone()
edge = torch.nonzero(changed[0] > 0)[0]
u, v = edge.tolist()
changed[0, u, v] = 0
changed[1, u, v] = 1
h_changed = rgcn_layer(
    relation_features, relation_normalize(changed), rel_weights, self_w
)
delta = (h_changed - h).norm(dim=1).detach()
assert float(delta.max()) > 0
plt.bar(np.arange(len(delta)), delta.numpy())
plt.xlabel("entity")
plt.ylabel("embedding change")
plt.title("edge-type intervention")
plt.show()
print("moved edge / max change:", (u, v), float(delta.max()))

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 7 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2) / Eq. (2)의 c_i,r / Eq. (3)
- **이 셀의 책임:** relation-specific neighborhood sum / 관계별 degree normalization / basis decomposition
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 관계별 weight·shape / 행합/isolated 처리 / parameter reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass

relation_adjacency = torch.tensor(raw["relation_adjacency"], dtype=torch.float32)
relation_features = torch.tensor(raw["relation_features"], dtype=torch.float32)


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 10
    basis_count: int = 2
    learning_rate: float = 0.03
    steps: int = 14


def preprocess_graph(relations: Tensor) -> Tensor:
    return relation_normalize(relations)


class RgcnPortfolioModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        relation_count: int,
        config: PortfolioConfig,
    ):
        super().__init__()
        self.config = config
        self.bases = nn.Parameter(
            torch.randn(config.basis_count, input_dim, config.hidden_dim) * 0.1
        )
        self.coefficients = nn.Parameter(
            torch.randn(relation_count, config.basis_count) * 0.1
        )
        self.self_weight = nn.Parameter(torch.randn(input_dim, config.hidden_dim) * 0.1)
        self.decoders = nn.Parameter(
            torch.randn(relation_count, config.hidden_dim, config.hidden_dim) * 0.1
        )

    def forward(self, inputs: Tensor, relations: Tensor) -> tuple[Tensor, Tensor]:
        relation_weights = basis_weights(self.bases, self.coefficients)
        hidden = rgcn_layer(
            inputs,
            relations,
            relation_weights,
            self.self_weight,
        )
        edge_logits = torch.einsum("ih,rhk,jk->rij", hidden, self.decoders, hidden)
        return hidden, edge_logits

    def compute_loss(self, edge_logits: Tensor, targets: Tensor) -> Tensor:
        return F.binary_cross_entropy_with_logits(edge_logits, targets)

    def training_step(
        self, inputs: Tensor, relations: Tensor, targets: Tensor
    ) -> Tensor:
        _, edge_logits = self(inputs, relations)
        return self.compute_loss(edge_logits, targets)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 8 · 목적함수·학습 update

- **원문 위치:** §2, Eq. (2) / Eq. (2)의 c_i,r / Eq. (3)
- **이 셀의 책임:** relation-specific neighborhood sum / 관계별 degree normalization / basis decomposition
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 관계별 weight·shape / 행합/isolated 처리 / parameter reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: RgcnPortfolioModel,
    inputs: Tensor,
    relations: Tensor,
    targets: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, relations, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 9 · 평가·완료 증거

- **원문 위치:** Eq. (3) / Figure 2
- **이 셀의 책임:** basis decomposition / edge type 변화의 메시지 영향
- **Tensor shape 계약:** relation A [R,N,N], X [N,D] → relation messages [N,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** parameter reconstruction / embedding displacement. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: RgcnPortfolioModel,
    inputs: Tensor,
    relations: Tensor,
    targets: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        _, logits = model(inputs, relations)
        probabilities = torch.sigmoid(logits)
        target_mask = targets.bool()
        gap = probabilities[target_mask].mean() - probabilities[~target_mask].mean()
        relation_weights = basis_weights(model.bases, model.coefficients)
    return {
        "relation_edge_gap": float(gap.cpu()),
        "basis_weight_norm": float(relation_weights.norm().cpu()),
    }


relation_features_device = ACCELERATOR.move(relation_features)
relation_adjacency_device = ACCELERATOR.move(relation_adjacency)
portfolio_relations = ACCELERATOR.move(preprocess_graph(relation_adjacency))
portfolio_model = ACCELERATOR.move(
    RgcnPortfolioModel(
        relation_features.shape[1],
        relation_adjacency.shape[0],
        PortfolioConfig(),
    )
)
portfolio_history = fit_portfolio_model(
    portfolio_model,
    relation_features_device,
    portfolio_relations,
    relation_adjacency_device,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    relation_features_device,
    portfolio_relations,
    relation_adjacency_device,
)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> AIFB/MUTAG/BGS/AM와 FB15k 대신 12개 entity·3 relation의 로컬 지식그래프를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.